In [22]:
!rm -rf NN-Project1/

In [23]:
!git clone https://www.github.com/yousefkoriem/NN-Project1.git

Cloning into 'NN-Project1'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 283 (delta 4), reused 35 (delta 3), pack-reused 247 (from 2)
Receiving objects: 100% (283/283), 231.58 MiB | 30.48 MiB/s, done.
Resolving deltas: 100% (64/64), done.


In [24]:
import os
os.chdir("/content/NN-Project1")

In [25]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from keras import layers
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle
from keras import layers, models, metrics, optimizers,callbacks

In [26]:
model = models.load_model("models/architecture/untrained_imdb_model.keras")

In [27]:
checkpoint = callbacks.ModelCheckpoint(
    filepath="models/architecture/best_imdb_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [28]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [29]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [30]:
# 1. Load the raw datasets
train_ds = tf.data.Dataset.load("data/processed/train_ds")
val_ds = tf.data.Dataset.load("data/processed/val_ds")
test_ds = tf.data.Dataset.load("data/processed/test_ds")

# 2. Define the label shape fix
def vectorize_text(text, label):
    return text, tf.expand_dims(label, -1)

# 3. Map the fix to the data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)

# 4. Apply cache and prefetch exactly ONCE at the very end
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [31]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
622/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6478 - f1_score: 0.6859 - loss: 0.5831
Epoch 1: val_loss improved from None to 0.31256, saving model to models/architecture/best_imdb_model.keras

Epoch 1: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7653 - f1_score: 0.7740 - loss: 0.4560 - val_accuracy: 0.8658 - val_f1_score: 0.8656 - val_loss: 0.3126 - learning_rate: 0.0010
Epoch 2/100
620/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8911 - f1_score: 0.8920 - loss: 0.2700
Epoch 2: val_loss did not improve from 0.31256
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9183 - f1_score: 0.9181 - loss: 0.2127 - val_accuracy: 0.8714 - val_f1_score: 0.8753 - val_loss: 0.3230 - learning_rate: 0.0010
Epoch 3/100
621/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9560 - f1_score: 0.9563 - loss: 0.1316
Epoch 3: val_loss did not improve from 0.31256
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms

In [ ]:
model.summary()

In [ ]:
!mkdir results/tables

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("results/tables/history.csv", index=False)

In [ ]:
score = model.evaluate(test_ds)
score_df = pd.DataFrame([score], columns=["loss", "accuracy", "f1_score"])

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8568 - f1_score: 0.8491 - loss: 0.3266


In [ ]:
!mkdir results/models

In [ ]:
model.save("results/models/final_imdb_model.keras")